In [1]:
import os  # for working with folders and file names
import cv2  # main computer vision library (OpenCV)
import numpy as np  # numerical operations (we may need later)
import matplotlib.pyplot as plt  # for showing images in the notebook

In [2]:
def remove_background(tile_rgb):
    """
    Removes background using adaptive threshold and contour masking.
    If the tile is too uniform (single color), return a clean white tile.
    """
    gray = cv2.cvtColor(tile_rgb, cv2.COLOR_RGB2GRAY)

    # --- NEW: detect uniform tiles ---
    if gray.std() < 5:  # very low texture / almost one color
        white_bg = np.ones_like(tile_rgb) * 255
        return white_bg

    # Adaptive threshold (for tiles that do have texture)
    thresh = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=15,
        C=10,
    )

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    mask = np.zeros_like(gray)
    cv2.drawContours(mask, contours, -1, 255, thickness=cv2.FILLED)

    tile_no_bg = cv2.bitwise_and(tile_rgb, tile_rgb, mask=mask)
    white_bg = np.ones_like(tile_rgb) * 255
    tile_clean = np.where(mask[:, :, np.newaxis] == 255, tile_no_bg, white_bg)

    return tile_clean

In [3]:
def process_puzzles(
    N,
    folder_name,
    canny_thresh1=10,
    canny_thresh2=40,
    use_equalize=True,
    use_gauss_for_canny=True,
):
    """
    Processes NxN puzzles: crops tiles, removes background, saves raw, preprocessed (gray+median),
    and edge (Canny) versions. Returns a list of tile records.

    Args:
        N (int): puzzle grid size (NxN)
        folder_name (str): relative folder under project where puzzle images live (e.g. "puzzle_2x2")
        canny_thresh1/canny_thresh2: thresholds for Canny (or use auto thresholds manually)
        use_equalize (bool): if True, run histogram equalization before Canny
        use_gauss_for_canny (bool): if True, apply GaussianBlur before Canny (recommended)
    """
    puzzle_folder = os.path.join("data", folder_name)

    raw_tiles_folder = os.path.join("output", f"tiles_{N}x{N}", "raw")
    prep_tiles_folder = os.path.join("output", f"tiles_{N}x{N}", "prep")
    edge_tiles_folder = os.path.join("output", f"tiles_{N}x{N}", "edges")

    os.makedirs(raw_tiles_folder, exist_ok=True)
    os.makedirs(prep_tiles_folder, exist_ok=True)
    os.makedirs(edge_tiles_folder, exist_ok=True)

    puzzle_files = sorted(
        f
        for f in os.listdir(puzzle_folder)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    )

    print(f"\nProcessing {N}x{N} puzzles from folder '{folder_name}'")
    print(f"Number of puzzles found: {len(puzzle_files)}")

    tile_records = []

    for puzzle_index, puzzle_name in enumerate(puzzle_files):
        print("\n==============================")
        print("Processing puzzle index:", puzzle_index, "file:", puzzle_name)
        print("==============================")

        puzzle_path = os.path.join(puzzle_folder, puzzle_name)
        puzzle_bgr = cv2.imread(puzzle_path)
        if puzzle_bgr is None:
            print(" WARNING: cannot read", puzzle_path)
            continue
        # convert to RGB for your remove_background which expects RGB
        puzzle_rgb = cv2.cvtColor(puzzle_bgr, cv2.COLOR_BGR2RGB)

        h, w, c = puzzle_rgb.shape
        print("  puzzle size:", h, "x", w, "channels:", c)

        tile_h, tile_w = h // N, w // N
        print("  tile size: ", tile_h, "x", tile_w)

        for r in range(N):
            for c in range(N):
                y1, y2 = r * tile_h, (r + 1) * tile_h
                x1, x2 = c * tile_w, (c + 1) * tile_w
                tile_rgb = puzzle_rgb[y1:y2, x1:x2].copy()

                tile_clean = remove_background(tile_rgb)

                tile_name = f"puz{puzzle_index}_tile{r}{c}.png"

                # 1. RAW (clean RGB)
                raw_path = os.path.join(raw_tiles_folder, tile_name)
                tile_clean_bgr = cv2.cvtColor(tile_clean, cv2.COLOR_RGB2BGR)
                cv2.imwrite(raw_path, tile_clean_bgr)

                # 2. PREP (grayscale + median blur)
                gray = cv2.cvtColor(tile_clean, cv2.COLOR_RGB2GRAY)
                blurred = cv2.medianBlur(gray, 5)
                prep_path = os.path.join(prep_tiles_folder, tile_name)
                cv2.imwrite(prep_path, blurred)

                # 3. EDGES (Canny)
                canny_input = blurred
                if use_equalize:
                    canny_input = cv2.equalizeHist(canny_input)
                if use_gauss_for_canny:
                    canny_input = cv2.GaussianBlur(canny_input, (5, 5), 0)

                edges = cv2.Canny(canny_input, canny_thresh1, canny_thresh2)
                edge_path = os.path.join(edge_tiles_folder, tile_name)
                cv2.imwrite(edge_path, edges)

                tile_records.append(
                    {
                        "puzzle_index": puzzle_index,
                        "row": r,
                        "col": c,
                        "raw_path": raw_path,
                        "prep_path": prep_path,
                        "edge_path": edge_path,
                    }
                )

    print(f"\n  Saved {len(tile_records)} tiles for {N}x{N} puzzles.")
    return tile_records

In [4]:
tiles_2x2 = process_puzzles(2, "puzzle_2x2")


Processing 2x2 puzzles from folder 'puzzle_2x2'
Number of puzzles found: 110

Processing puzzle index: 0 file: 0.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 1 file: 1.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 2 file: 10.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 3 file: 100.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 4 file: 101.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 5 file: 102.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 6 file: 103.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 7 file: 104.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112

Processing puzzle index: 8 file: 105.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  112 x 112


In [5]:
tiles_4x4 = process_puzzles(4, "puzzle_4x4")


Processing 4x4 puzzles from folder 'puzzle_4x4'
Number of puzzles found: 110

Processing puzzle index: 0 file: 0.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56



Processing puzzle index: 1 file: 1.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 2 file: 10.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 3 file: 100.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 4 file: 101.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 5 file: 102.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 6 file: 103.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 7 file: 104.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 6 file: 103.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 7 file: 104.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  56 x 56

Processing puzzle index: 8 file: 105.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  5

In [6]:
tiles_8x8 = process_puzzles(8, "puzzle_8x8")


Processing 8x8 puzzles from folder 'puzzle_8x8'
Number of puzzles found: 110

Processing puzzle index: 0 file: 0.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 1 file: 1.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 1 file: 1.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 2 file: 10.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 3 file: 100.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 2 file: 10.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 3 file: 100.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 4 file: 101.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle index: 4 file: 101.jpg
  puzzle size: 224 x 224 channels: 3
  tile size:  28 x 28

Processing puzzle in

In [7]:
def visualize_tiles(tile_records, N, num_puzzles_to_show=5):
    """
    Visualize puzzles with their tiles before and after preprocessing.
    Top row: RAW, Bottom row: PREP (grayscale + median filter)
    """
    for check_index in range(num_puzzles_to_show):
        # Get tiles for this puzzle
        records = [rec for rec in tile_records if rec["puzzle_index"] == check_index]
        records = sorted(records, key=lambda rec: (rec["row"], rec["col"]))
        num_tiles = len(records)
        if num_tiles == 0:
            print(f"No tiles found for puzzle index {check_index}")
            continue

        print(f"\nShowing puzzle index {check_index} with {num_tiles} tiles")

        fig, axes = plt.subplots(2, num_tiles, figsize=(num_tiles * 2, 4))

        # Flatten axes to 1D array for easy indexing
        axes = np.array(axes).reshape(2, num_tiles)

        for i, rec in enumerate(records):
            # Load raw
            raw_bgr = cv2.imread(rec["raw_path"])
            raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

            # Load preprocessed
            prep_gray = cv2.imread(rec["prep_path"], cv2.IMREAD_GRAYSCALE)

            # Top row: RAW
            axes[0, i].imshow(raw_rgb)
            axes[0, i].axis("off")
            if i == 0:
                axes[0, i].set_title("RAW")

            # Bottom row: PREP
            axes[1, i].imshow(prep_gray, cmap="gray")
            axes[1, i].axis("off")
            if i == 0:
                axes[1, i].set_title(
                    "PREP (background removal +gray + median+canny edge )"
                )

        plt.suptitle(f"Puzzle {check_index:03d} – tiles before & after pipeline")
        plt.tight_layout()
        plt.show()

In [8]:
# Visualize 2x2 puzzles
visualize_tiles(x2, N=2)

NameError: name 'x2' is not defined

In [ ]:
# Visualize 4x4 puzzles
visualize_tiles(tiles_4x4, N=4)

In [ ]:
# Visualize 8x8 puzzles
visualize_tiles(tiles_8x8, N=8)